# Reinforcement Learning for Rule-Following: Amazon Review Compliance Chatbot

This notebook builds an **RL-trained chatbot** that learns to generate Amazon reviews that comply with Amazon's Community Guidelines — from scratch, using only NumPy.

## What you will learn

| Section | Topic |
|---------|-------|
| 1 | RL Framework for Language Models |
| 2 | Amazon Review Rules → Reward Functions |
| 3 | Policy: Token-level Generative Model |
| 4 | REINFORCE (Policy Gradient) |
| 5 | PPO (Proximal Policy Optimization) |
| 6 | GRPO (Group Relative Policy Optimization — DeepSeek R1 style) |
| 7 | Training Demo with Before/After Comparison |
| 8 | Rule Compliance Evaluation |
| 9 | Scaling to Real LLMs (RLHF, Constitutional AI) |

**Key insight**: Instead of labelling millions of "good" vs "bad" reviews and doing supervised learning, RL lets the model discover what makes a review compliant by *trying* and receiving rule-based reward signals.

## Section 1: Why RL for Rule Following?

### Supervised Learning Limitations
- Requires labelled examples of every possible rule violation
- Rules change → need to re-label data
- Cannot handle rules that are hard to demonstrate (e.g., "don't be deceptive")

### RL Advantages
- Rules become **reward functions** — no labelled data needed
- Add a new rule → write a new reward component
- The model discovers *how* to satisfy rules, not just *that* it should

### The RL MDP for Text Generation

```
State  s_t  = tokens generated so far (context)
Action a_t  = next token to emit
Policy π(a|s) = probability distribution over vocabulary
Reward R     = scalar score at end of episode (full review generated)
Episode      = generating one complete review (start → EOS token)
```

The agent (language model) generates token-by-token until `<EOS>`, then receives a reward based on whether the full review complies with Amazon's rules.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict, Counter
import re
import random
from copy import deepcopy

np.random.seed(42)
random.seed(42)
print('Libraries loaded. Pure NumPy RL — no ML frameworks required.')

## Section 2: Amazon Review Rules → Reward Functions

Amazon's [Community Guidelines](https://www.amazon.com/gp/help/customer/display.html?nodeId=GLHXEX85MENUE4XF) prohibit:

| Rule | Why it exists |
|------|---------------|
| No promotional/advertising content | Maintains trust |
| No URLs or links | Prevents spam |
| No profanity or hate speech | Community standards |
| No personally identifiable info | Privacy |
| No off-topic content (must be about the product) | Relevance |
| No incentivized reviews ("got this for free") | Authenticity |
| Minimum length (substantive content) | Quality |
| No ALL-CAPS | Readability |
| No duplicate/repetitive text | Quality |

Each rule becomes a **reward component**. The total reward is a weighted sum.

In [ ]:
# ─────────────────────────────────────────────
# Amazon Review Rule-Based Reward Functions
# ─────────────────────────────────────────────

PROFANITY = {'damn', 'hell', 'crap', 'junk', 'garbage', 'scam', 'fraud', 'fake', 'hate', 'terrible'}
PROMO_WORDS = {'buy', 'purchase', 'click', 'discount', 'offer', 'sale', 'coupon', 'promo', 'sponsored', 'advertisement'}
INCENTIVE_PHRASES = ['got this for free', 'received this for free', 'free product', 'discount in exchange',
                     'got it free', 'complimentary', 'provided free']
PRODUCT_WORDS = {'product', 'item', 'quality', 'works', 'great', 'good', 'bad', 'price', 'value',
                 'recommend', 'love', 'use', 'excellent', 'satisfied', 'happy', 'pleased', 'worth',
                 'fast', 'delivery', 'shipping', 'arrived', 'package', 'durable', 'material', 'size',
                 'color', 'easy', 'problem', 'issue', 'return', 'customer', 'service', 'perfect',
                 'disappointed', 'broken', 'damaged', 'well', 'made', 'design', 'comfortable', 'fits'}


def reward_no_profanity(text: str) -> float:
    """Penalty for each profane word used."""
    words = set(text.lower().split())
    violations = words & PROFANITY
    return -0.3 * len(violations)


def reward_no_promotional(text: str) -> float:
    """Penalty for promotional / advertising language."""
    words = set(text.lower().split())
    violations = words & PROMO_WORDS
    return -0.5 * len(violations)


def reward_no_urls(text: str) -> float:
    """Penalty for any URL-like content."""
    has_url = bool(re.search(r'https?://|www\.|\.(com|net|org)\b', text, re.I))
    return -1.0 if has_url else 0.0


def reward_no_incentivized(text: str) -> float:
    """Penalty for disclosing receiving free products (used to game the system)."""
    t = text.lower()
    for phrase in INCENTIVE_PHRASES:
        if phrase in t:
            return -1.5
    return 0.0


def reward_minimum_length(text: str, min_words: int = 15) -> float:
    """Reward substantive reviews; penalise ultra-short ones."""
    n = len(text.split())
    if n < 5:
        return -1.0
    if n < min_words:
        return -0.5 + (n / min_words) * 0.5
    return 0.5  # Base quality bonus


def reward_no_allcaps(text: str) -> float:
    """Penalise excessive ALL-CAPS usage."""
    words = text.split()
    if not words:
        return 0.0
    caps_ratio = sum(1 for w in words if w.isupper() and len(w) > 2) / len(words)
    return -caps_ratio  # Proportional penalty


def reward_on_topic(text: str) -> float:
    """Reward for content that mentions product-related concepts."""
    words = set(text.lower().split())
    topic_hits = words & PRODUCT_WORDS
    score = min(len(topic_hits) * 0.1, 1.0)  # Capped at 1.0
    return score


def reward_no_repetition(text: str) -> float:
    """Penalise repetitive phrases."""
    words = text.lower().split()
    if len(words) < 4:
        return 0.0
    bigrams = list(zip(words, words[1:]))
    bigram_counts = Counter(bigrams)
    repetitions = sum(c - 1 for c in bigram_counts.values() if c > 1)
    return -0.2 * repetitions


REWARD_WEIGHTS = {
    'no_profanity':    1.0,
    'no_promotional':  1.0,
    'no_urls':         1.0,
    'no_incentivized': 1.0,
    'length':          1.0,
    'no_allcaps':      0.5,
    'on_topic':        2.0,  # Most important: review must be about the product
    'no_repetition':   0.5,
}


def compute_reward(text: str, verbose: bool = False) -> float:
    """Compute total Amazon-compliance reward for a review text."""
    components = {
        'no_profanity':    reward_no_profanity(text),
        'no_promotional':  reward_no_promotional(text),
        'no_urls':         reward_no_urls(text),
        'no_incentivized': reward_no_incentivized(text),
        'length':          reward_minimum_length(text),
        'no_allcaps':      reward_no_allcaps(text),
        'on_topic':        reward_on_topic(text),
        'no_repetition':   reward_no_repetition(text),
    }
    total = sum(REWARD_WEIGHTS[k] * v for k, v in components.items())
    if verbose:
        print(f'Reward breakdown for: "{text[:60]}..."' if len(text) > 60 else f'Reward breakdown for: "{text}"')
        for k, v in components.items():
            weighted = REWARD_WEIGHTS[k] * v
            bar = '█' * int(abs(weighted) * 5) if weighted != 0 else '·'
            sign = '+' if weighted >= 0 else ''
            print(f'  {k:<20} {sign}{weighted:+.2f}  {bar}')
        print(f'  {"TOTAL":<20} {total:+.3f}')
    return total


# Test the reward function on example reviews
bad_review_1 = 'Buy NOW! CLICK HERE www.spam.com Best deal ever! Terrible scam junk garbage hate'
bad_review_2 = 'Got this for free in exchange for review. OK product.'
good_review = ('This product works great and the quality is excellent. '
               'Fast shipping, arrived well packaged. I would recommend it '
               'to anyone looking for good value. Very satisfied with the purchase.')

print('=== Reward Function Tests ===')
for name, text in [('BAD (spam)', bad_review_1), ('BAD (incentivized)', bad_review_2), ('GOOD', good_review)]:
    r = compute_reward(text)
    print(f'\n[{name}] Reward: {r:.3f}')
    compute_reward(text, verbose=True)

## Section 3: The Policy Model

We build a **bigram language model** as the policy — simple enough to train from scratch, but rich enough to demonstrate RL principles.

The policy is parameterised as:
```
π_θ(token | prev_token) = softmax(W[prev_token])  # W ∈ R^{V×V}
```

At each step, given the last token, the policy samples the next token from a softmax distribution. This is the **action** in our MDP.

In [ ]:
# ─────────────────────────────────────────────
# Vocabulary: words the model can generate
# ─────────────────────────────────────────────

GOOD_WORDS = [
    '<BOS>', '<EOS>',
    'the', 'this', 'a', 'an', 'is', 'was', 'are', 'i', 'my', 'we',
    'product', 'item', 'quality', 'material', 'design', 'size', 'color',
    'works', 'arrived', 'fits', 'holds', 'feels', 'looks', 'lasts',
    'great', 'good', 'excellent', 'perfect', 'nice', 'solid', 'sturdy',
    'pleased', 'satisfied', 'happy', 'impressed', 'love', 'recommend',
    'fast', 'quick', 'well', 'packaged', 'delivery', 'shipping',
    'worth', 'value', 'price', 'affordable', 'durable', 'comfortable',
    'would', 'will', 'definitely', 'highly', 'really', 'very', 'so',
    'easy', 'use', 'install', 'clean', 'made', 'built',
    'expected', 'described', 'advertised', 'pictures', 'same',
]

BAD_WORDS = [
    'buy', 'click', 'discount', 'sale', 'coupon', 'promo', 'sponsored',  # promotional
    'scam', 'fake', 'junk', 'garbage', 'terrible', 'hate', 'crap', 'damn',  # profanity
    'free', 'complimentary', 'exchange',  # incentivized
]

VOCAB = GOOD_WORDS + BAD_WORDS
V = len(VOCAB)
token2id = {w: i for i, w in enumerate(VOCAB)}
id2token = {i: w for w, i in token2id.items()}

BOS_ID = token2id['<BOS>']
EOS_ID = token2id['<EOS>']

print(f'Vocabulary size: {V} tokens')
print(f'Good words: {len(GOOD_WORDS)}, Bad words (violations): {len(BAD_WORDS)}')


def softmax(x: np.ndarray) -> np.ndarray:
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()


class BigramPolicy:
    """
    Bigram language model policy: π(next_token | prev_token) = softmax(W[prev_token]).
    
    Parameters:
        W: (V, V) logit matrix. W[i, j] = log-odds of emitting token j given token i.
    """
    def __init__(self, vocab_size: int, temperature: float = 1.0):
        self.V = vocab_size
        self.temperature = temperature
        # Initialise with small random logits — uniform-ish distribution
        self.W = np.random.randn(vocab_size, vocab_size) * 0.1
        # Discourage EOS at the start
        self.W[:, EOS_ID] -= 3.0

    def log_probs(self, prev_token_id: int) -> np.ndarray:
        """Log-probabilities for next token given prev token."""
        logits = self.W[prev_token_id] / self.temperature
        probs = softmax(logits)
        return np.log(probs + 1e-10)

    def probs(self, prev_token_id: int) -> np.ndarray:
        return np.exp(self.log_probs(prev_token_id))

    def sample(self, prev_token_id: int) -> int:
        """Sample next token from the policy distribution."""
        p = self.probs(prev_token_id)
        return np.random.choice(self.V, p=p)

    def generate(self, max_len: int = 30) -> tuple[list[int], list[float]]:
        """
        Generate a sequence of tokens and record log-probs (needed for REINFORCE).
        Returns: (token_ids, log_probs_of_chosen_tokens)
        """
        tokens = [BOS_ID]
        log_probs_taken = []
        for _ in range(max_len):
            prev = tokens[-1]
            lp = self.log_probs(prev)
            next_tok = np.random.choice(self.V, p=np.exp(lp))
            tokens.append(next_tok)
            log_probs_taken.append(lp[next_tok])
            if next_tok == EOS_ID:
                break
        return tokens, log_probs_taken

    def tokens_to_text(self, token_ids: list[int]) -> str:
        """Convert token IDs back to a readable string."""
        return ' '.join(id2token[t] for t in token_ids if t not in (BOS_ID, EOS_ID))

    def clone(self) -> 'BigramPolicy':
        p = BigramPolicy(self.V, self.temperature)
        p.W = self.W.copy()
        return p


# Demonstrate an untrained policy
untrained = BigramPolicy(V)
tokens, lps = untrained.generate(max_len=20)
text = untrained.tokens_to_text(tokens)
reward = compute_reward(text)
print(f'Untrained policy sample: "{text}"')
print(f'Reward: {reward:.3f}')

## Section 4: REINFORCE — Policy Gradient

REINFORCE (Williams 1992) is the simplest policy gradient algorithm.

### The Policy Gradient Theorem
For an episode with tokens $a_1, ..., a_T$ and final reward $R$:

$$\nabla_\theta J(\theta) = \mathbb{E}\left[ R \cdot \sum_{t=1}^{T} \nabla_\theta \log \pi_\theta(a_t | s_t) \right]$$

**Intuition**: Increase probabilities of actions taken in high-reward episodes, decrease them for low-reward episodes.

### With Baseline (Variance Reduction)
Subtracting a baseline $b$ (moving average of rewards) reduces variance without changing the gradient in expectation:

$$\nabla_\theta J(\theta) = \mathbb{E}\left[ (R - b) \cdot \sum_{t=1}^{T} \nabla_\theta \log \pi_\theta(a_t | s_t) \right]$$

In [ ]:
# ─────────────────────────────────────────────
# REINFORCE Algorithm
# ─────────────────────────────────────────────

def compute_policy_gradient_reinforce(policy: BigramPolicy,
                                       tokens: list[int],
                                       reward: float,
                                       baseline: float) -> np.ndarray:
    """
    Compute the REINFORCE gradient for the policy W matrix.
    
    For each step t:
        ∂J/∂W[prev_t, :] += (R - b) * (1{a_t} - π(·|prev_t))
    
    This is the score function estimator (log-derivative trick):
        ∇_θ log π_θ(a|s) = 1_a - π_θ(·|s)  [for softmax]
    """
    grad_W = np.zeros_like(policy.W)
    advantage = reward - baseline

    for t in range(len(tokens) - 1):
        prev_tok = tokens[t]
        next_tok = tokens[t + 1]
        if next_tok == BOS_ID:
            continue

        # π_θ(·|prev_tok): current probability distribution
        p = policy.probs(prev_tok)

        # Score function: ∇ log π = one_hot(action) - π
        one_hot = np.zeros(policy.V)
        one_hot[next_tok] = 1.0
        score = one_hot - p  # (V,)

        # Accumulate gradient
        grad_W[prev_tok] += advantage * score

    return grad_W


class REINFORCE:
    """REINFORCE trainer with exponential moving average baseline."""

    def __init__(self, policy: BigramPolicy, lr: float = 0.01, baseline_decay: float = 0.99):
        self.policy = policy
        self.lr = lr
        self.baseline = 0.0
        self.baseline_decay = baseline_decay
        self.reward_history = []
        self.step_count = 0

    def step(self, max_len: int = 25) -> dict:
        """One REINFORCE update step: sample episode → compute reward → update policy."""
        # Sample an episode
        tokens, log_probs = self.policy.generate(max_len=max_len)
        text = self.policy.tokens_to_text(tokens)
        reward = compute_reward(text)

        # Compute gradient
        grad_W = compute_policy_gradient_reinforce(self.policy, tokens, reward, self.baseline)

        # Gradient ascent (maximise reward)
        self.policy.W += self.lr * grad_W

        # Update baseline (EMA of rewards)
        self.baseline = self.baseline_decay * self.baseline + (1 - self.baseline_decay) * reward
        self.reward_history.append(reward)
        self.step_count += 1

        return {'reward': reward, 'text': text, 'tokens': tokens}


print('REINFORCE algorithm defined.')
print('Core update: W += lr * (R - baseline) * (one_hot(action) - π(action|state))')

## Section 5: PPO — Proximal Policy Optimization

PPO (Schulman et al. 2017) improves on REINFORCE by preventing catastrophically large policy updates.

### Clipped Objective
$$L^{CLIP}(\theta) = \mathbb{E}\left[ \min\left( r_t(\theta) \hat{A}_t,\ \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]$$

Where $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ is the probability ratio.

**Key idea**: If the policy wants to change too much (ratio far from 1), clip the gradient — stay close to the previous policy.

In [ ]:
# ─────────────────────────────────────────────
# PPO with Clipped Objective
# ─────────────────────────────────────────────

class PPO:
    """
    Simplified PPO for the bigram policy.
    Collects a mini-batch of episodes, then does K epochs of clipped updates.
    """

    def __init__(self, policy: BigramPolicy, lr: float = 0.01,
                 clip_eps: float = 0.2, n_epochs: int = 4,
                 batch_size: int = 8):
        self.policy = policy
        self.lr = lr
        self.clip_eps = clip_eps
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.baseline = 0.0
        self.reward_history = []
        self.step_count = 0

    def collect_episodes(self, max_len: int = 25) -> list[dict]:
        """Collect a batch of episodes using the current policy."""
        episodes = []
        for _ in range(self.batch_size):
            tokens, _ = self.policy.generate(max_len=max_len)
            text = self.policy.tokens_to_text(tokens)
            reward = compute_reward(text)

            # Record old log-probs for importance sampling
            old_log_probs = []
            for t in range(len(tokens) - 1):
                prev_tok = tokens[t]
                next_tok = tokens[t + 1]
                old_log_probs.append(self.policy.log_probs(prev_tok)[next_tok])

            episodes.append({'tokens': tokens, 'text': text,
                             'reward': reward, 'old_log_probs': old_log_probs})
        return episodes

    def ppo_update(self, episodes: list[dict]) -> float:
        """Perform K epochs of PPO clipped updates over collected episodes."""
        rewards = np.array([e['reward'] for e in episodes])
        self.baseline = 0.95 * self.baseline + 0.05 * rewards.mean()
        advantages = rewards - self.baseline

        total_loss = 0.0
        for _ in range(self.n_epochs):
            grad_W = np.zeros_like(self.policy.W)

            for ep, adv in zip(episodes, advantages):
                tokens = ep['tokens']
                old_lps = ep['old_log_probs']

                for t in range(len(tokens) - 1):
                    prev_tok = tokens[t]
                    next_tok = tokens[t + 1]
                    if next_tok == BOS_ID or t >= len(old_lps):
                        continue

                    # Current log-prob
                    curr_lp = self.policy.log_probs(prev_tok)[next_tok]
                    old_lp = old_lps[t]

                    # Probability ratio r = π_new / π_old
                    ratio = np.exp(curr_lp - old_lp)

                    # Clipped PPO objective gradient
                    # Unclipped: ratio * advantage
                    # Clipped: clip(ratio, 1-eps, 1+eps) * advantage
                    clipped_ratio = np.clip(ratio, 1 - self.clip_eps, 1 + self.clip_eps)

                    if adv >= 0:
                        # For positive advantage: use min(unclipped, clipped)
                        # Gradient is blocked if ratio > 1+eps
                        effective_ratio = min(ratio, clipped_ratio)
                    else:
                        # For negative advantage: use max(unclipped, clipped)
                        # Gradient is blocked if ratio < 1-eps
                        effective_ratio = max(ratio, clipped_ratio)

                    # Score function gradient scaled by effective ratio
                    p = self.policy.probs(prev_tok)
                    one_hot = np.zeros(self.policy.V)
                    one_hot[next_tok] = 1.0
                    score = one_hot - p

                    grad_W[prev_tok] += adv * effective_ratio * score
                    total_loss += min(ratio * adv, clipped_ratio * adv)

            self.policy.W += self.lr * grad_W / (self.batch_size * self.n_epochs)

        return total_loss

    def step(self, max_len: int = 25) -> dict:
        """One PPO step: collect batch → update policy K times."""
        episodes = self.collect_episodes(max_len)
        self.ppo_update(episodes)

        avg_reward = np.mean([e['reward'] for e in episodes])
        best_ep = max(episodes, key=lambda e: e['reward'])
        self.reward_history.append(avg_reward)
        self.step_count += 1

        return {'reward': avg_reward, 'text': best_ep['text']}


print('PPO defined. Key improvement over REINFORCE:')
print('  - Collects batch_size episodes before each update')
print('  - Does n_epochs gradient steps per batch')
print('  - Clips probability ratio to prevent destabilising large updates')

## Section 6: GRPO — Group Relative Policy Optimization

GRPO (Shao et al. 2024) is used in **DeepSeek-R1** to teach reasoning. It eliminates the need for a learned value function by using **group-relative advantages**.

### Key Idea
For each prompt, generate **G samples** (a group). Advantage = how this sample compares to the **group mean**:

$$\hat{A}_i = \frac{R_i - \text{mean}(R_1, ..., R_G)}{\text{std}(R_1, ..., R_G)}$$

This is a **self-normalising** advantage — no separate value/critic network needed.

### Why it works for rule-following
- Generate G reviews for the same product category
- Reviews that better follow Amazon rules get positive advantage
- Reviews that violate rules get negative advantage
- Policy updates amplify the rule-following behaviour

In [ ]:
# ─────────────────────────────────────────────
# GRPO — Group Relative Policy Optimization
# ─────────────────────────────────────────────

class GRPO:
    """
    Group Relative Policy Optimization (DeepSeek-R1 style).
    
    For each step:
      1. Sample G episodes (a group)
      2. Compute rewards R_1, ..., R_G
      3. Normalise: A_i = (R_i - mean(R)) / (std(R) + eps)
      4. Gradient update with clipped importance sampling
      5. KL penalty: keep new policy close to reference policy
    """

    def __init__(self, policy: BigramPolicy, lr: float = 0.01,
                 group_size: int = 8, clip_eps: float = 0.2,
                 kl_coef: float = 0.01):
        self.policy = policy
        self.ref_policy = policy.clone()  # Reference policy for KL penalty
        self.lr = lr
        self.group_size = group_size
        self.clip_eps = clip_eps
        self.kl_coef = kl_coef
        self.reward_history = []
        self.step_count = 0

    def compute_kl_gradient(self, prev_tok: int) -> np.ndarray:
        """
        Gradient of KL(π_new || π_ref) w.r.t. W[prev_tok].
        KL(P||Q) = Σ P log(P/Q)
        ∂KL/∂logits_new ∝ π_new - π_ref
        """
        p_new = self.policy.probs(prev_tok)
        p_ref = self.ref_policy.probs(prev_tok)
        # Gradient of KL w.r.t. softmax logits
        return p_new - p_ref

    def step(self, max_len: int = 25) -> dict:
        """One GRPO step: sample group → normalise advantages → update."""
        # Step 1: Sample a group of G episodes
        episodes = []
        for _ in range(self.group_size):
            tokens, _ = self.policy.generate(max_len=max_len)
            text = self.policy.tokens_to_text(tokens)
            reward = compute_reward(text)

            old_log_probs = []
            for t in range(len(tokens) - 1):
                lp = self.policy.log_probs(tokens[t])[tokens[t + 1]]
                old_log_probs.append(lp)

            episodes.append({'tokens': tokens, 'text': text,
                             'reward': reward, 'old_log_probs': old_log_probs})

        # Step 2-3: Group-relative advantage normalisation
        rewards = np.array([e['reward'] for e in episodes])
        mu, sigma = rewards.mean(), rewards.std() + 1e-8
        advantages = (rewards - mu) / sigma

        # Step 4: Policy gradient with clipping + KL penalty
        grad_W = np.zeros_like(self.policy.W)

        for ep, adv in zip(episodes, advantages):
            tokens = ep['tokens']
            old_lps = ep['old_log_probs']

            for t in range(len(tokens) - 1):
                prev_tok = tokens[t]
                next_tok = tokens[t + 1]
                if next_tok == BOS_ID or t >= len(old_lps):
                    continue

                # Importance sampling ratio
                curr_lp = self.policy.log_probs(prev_tok)[next_tok]
                ratio = np.exp(curr_lp - old_lps[t])
                clipped_ratio = np.clip(ratio, 1 - self.clip_eps, 1 + self.clip_eps)
                eff_ratio = min(ratio, clipped_ratio) if adv >= 0 else max(ratio, clipped_ratio)

                # Policy gradient
                p = self.policy.probs(prev_tok)
                one_hot = np.zeros(self.policy.V)
                one_hot[next_tok] = 1.0
                pg_grad = adv * eff_ratio * (one_hot - p)

                # KL penalty gradient (subtract: penalise divergence from reference)
                kl_grad = -self.kl_coef * self.compute_kl_gradient(prev_tok)

                grad_W[prev_tok] += pg_grad + kl_grad

        self.policy.W += self.lr * grad_W / self.group_size

        avg_reward = rewards.mean()
        best_ep = max(episodes, key=lambda e: e['reward'])
        self.reward_history.append(avg_reward)
        self.step_count += 1

        return {
            'reward': avg_reward,
            'reward_std': rewards.std(),
            'best_reward': rewards.max(),
            'text': best_ep['text'],
            'advantages': advantages,
        }


print('GRPO defined.')
print('Comparison of algorithms:')
print('  REINFORCE: 1 episode/step, raw reward, no clipping')
print('  PPO:       B episodes/step, EMA baseline, clipped ratio, K epochs')
print('  GRPO:      G episodes/step, group-normalised advantage, clipped ratio, KL penalty')

## Section 7: Training Demo

Train all three algorithms and compare their learning curves.

In [ ]:
# ─────────────────────────────────────────────
# Training: REINFORCE vs PPO vs GRPO
# ─────────────────────────────────────────────

N_STEPS = 500  # Training steps per algorithm
SMOOTH_K = 20  # Smoothing window for plotting

def smooth(arr, k):
    return np.convolve(arr, np.ones(k) / k, mode='valid')


def sample_reviews(policy, n=5, max_len=25):
    """Sample n reviews and return with rewards."""
    results = []
    for _ in range(n):
        tokens, _ = policy.generate(max_len=max_len)
        text = policy.tokens_to_text(tokens)
        r = compute_reward(text)
        results.append((r, text))
    return sorted(results, reverse=True)


print('=== Before Training (random policy) ===')
init_policy = BigramPolicy(V)
for r, t in sample_reviews(init_policy, n=3):
    print(f'  [{r:+.2f}] {t}')
print()

# Train REINFORCE
print('Training REINFORCE...', end=' ', flush=True)
reinforce = REINFORCE(BigramPolicy(V), lr=0.02)
for s in range(N_STEPS):
    reinforce.step()
print(f'done. Final avg reward: {np.mean(reinforce.reward_history[-50:]):.3f}')

# Train PPO
print('Training PPO...       ', end=' ', flush=True)
ppo = PPO(BigramPolicy(V), lr=0.02, clip_eps=0.2, n_epochs=4, batch_size=8)
for s in range(N_STEPS // 8):  # Each step processes 8 episodes
    ppo.step()
ppo.reward_history = np.repeat(ppo.reward_history, 8)[:N_STEPS].tolist()
print(f'done. Final avg reward: {np.mean(ppo.reward_history[-50:]):.3f}')

# Train GRPO
print('Training GRPO...      ', end=' ', flush=True)
grpo = GRPO(BigramPolicy(V), lr=0.02, group_size=8, clip_eps=0.2, kl_coef=0.01)
for s in range(N_STEPS // 8):
    grpo.step()
grpo.reward_history = np.repeat(grpo.reward_history, 8)[:N_STEPS].tolist()
print(f'done. Final avg reward: {np.mean(grpo.reward_history[-50:]):.3f}')

print()
print('=== After Training — REINFORCE policy ===')
for r, t in sample_reviews(reinforce.policy, n=3):
    print(f'  [{r:+.2f}] {t}')

print()
print('=== After Training — GRPO policy ===')
for r, t in sample_reviews(grpo.policy, n=3):
    print(f'  [{r:+.2f}] {t}')

In [ ]:
# ─────────────────────────────────────────────
# Learning Curve Comparison
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = {'REINFORCE': '#e74c3c', 'PPO': '#3498db', 'GRPO': '#2ecc71'}

for name, hist in [('REINFORCE', reinforce.reward_history),
                   ('PPO', ppo.reward_history),
                   ('GRPO', grpo.reward_history)]:
    s = smooth(hist, SMOOTH_K)
    ax.plot(s, label=name, color=colors[name], linewidth=2)

ax.axhline(0, color='gray', linestyle='--', alpha=0.5, label='Baseline (0)')
ax.set_xlabel('Training Step')
ax.set_ylabel('Average Reward')
ax.set_title('Learning Curves: REINFORCE vs PPO vs GRPO')
ax.legend()
ax.grid(alpha=0.3)

# Bar chart: final 50-step average rewards
ax2 = axes[1]
final_rewards = {
    'REINFORCE': np.mean(reinforce.reward_history[-50:]),
    'PPO': np.mean(ppo.reward_history[-50:]),
    'GRPO': np.mean(grpo.reward_history[-50:]),
}
bars = ax2.bar(final_rewards.keys(), final_rewards.values(),
               color=[colors[k] for k in final_rewards], alpha=0.85, edgecolor='black')
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars, final_rewards.values()):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
ax2.set_ylabel('Avg Reward (last 50 steps)')
ax2.set_title('Final Performance Comparison')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('rl_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: rl_learning_curves.png')

## Section 8: Rule Compliance Evaluation

Evaluate how much each algorithm's trained policy improves compliance with each Amazon rule.

In [ ]:
# ─────────────────────────────────────────────
# Per-Rule Compliance Analysis
# ─────────────────────────────────────────────

RULE_FNS = {
    'No Profanity':     reward_no_profanity,
    'No Promotional':   reward_no_promotional,
    'No URLs':          reward_no_urls,
    'No Incentivized':  reward_no_incentivized,
    'Min Length':       reward_minimum_length,
    'No ALL-CAPS':      reward_no_allcaps,
    'On Topic':         reward_on_topic,
    'No Repetition':    reward_no_repetition,
}


def evaluate_policy(policy, n_samples=100, max_len=25):
    """Evaluate per-rule compliance across n_samples generated reviews."""
    rule_scores = defaultdict(list)
    total_rewards = []

    for _ in range(n_samples):
        tokens, _ = policy.generate(max_len=max_len)
        text = policy.tokens_to_text(tokens)
        total_rewards.append(compute_reward(text))
        for rule_name, fn in RULE_FNS.items():
            rule_scores[rule_name].append(fn(text))

    return {
        'total': np.mean(total_rewards),
        'rules': {k: np.mean(v) for k, v in rule_scores.items()},
    }


print('Evaluating policies (100 samples each)...')
baseline_eval = evaluate_policy(init_policy)
reinforce_eval = evaluate_policy(reinforce.policy)
ppo_eval = evaluate_policy(ppo.policy)
grpo_eval = evaluate_policy(grpo.policy)

evals = {
    'Random (untrained)': baseline_eval,
    'REINFORCE': reinforce_eval,
    'PPO': ppo_eval,
    'GRPO': grpo_eval,
}

# Print table
print(f'\n{"Rule":<22}', end='')
for name in evals:
    print(f'{name:>18}', end='')
print()
print('-' * (22 + 18 * len(evals)))

for rule in RULE_FNS:
    print(f'{rule:<22}', end='')
    for ev in evals.values():
        val = ev['rules'][rule]
        print(f'{val:>+18.3f}', end='')
    print()

print('-' * (22 + 18 * len(evals)))
print(f'{"TOTAL REWARD":<22}', end='')
for ev in evals.values():
    print(f'{ev["total"]:>+18.3f}', end='')
print()

# Heatmap
fig, ax = plt.subplots(figsize=(12, 6))
rule_names = list(RULE_FNS.keys())
algo_names = list(evals.keys())
data = np.array([[ev['rules'][r] for r in rule_names] for ev in evals.values()])

im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=-0.5, vmax=1.0)
ax.set_xticks(range(len(rule_names)))
ax.set_xticklabels(rule_names, rotation=30, ha='right')
ax.set_yticks(range(len(algo_names)))
ax.set_yticklabels(algo_names)
for i in range(len(algo_names)):
    for j in range(len(rule_names)):
        ax.text(j, i, f'{data[i, j]:+.2f}', ha='center', va='center', fontsize=9, fontweight='bold')
plt.colorbar(im, ax=ax, label='Rule Score')
ax.set_title('Per-Rule Compliance: Random vs Trained Policies')
plt.tight_layout()
plt.savefig('rl_rule_compliance.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 9: Interactive Chatbot Demo

Simulate the chatbot use case: the user provides a **product context**, the trained model generates a compliant review, and we show rule-by-rule feedback.

In [ ]:
# ─────────────────────────────────────────────
# Amazon Review Compliance Chatbot
# ─────────────────────────────────────────────

class AmazonReviewChatbot:
    """
    Chatbot that:
      1. Takes a user-submitted review draft
      2. Checks it against Amazon rules
      3. Explains violations
      4. Generates a compliant alternative using the RL-trained policy
    """

    RULE_EXPLANATIONS = {
        'No Profanity': 'Amazon prohibits offensive language in reviews.',
        'No Promotional': 'Reviews cannot contain advertising or promotional content.',
        'No URLs': 'External links and URLs are not allowed.',
        'No Incentivized': 'Do not disclose receiving the product free/discounted for review.',
        'Min Length': 'Reviews should be substantive (at least 15 words).',
        'No ALL-CAPS': 'Avoid excessive capitalization — it reads as shouting.',
        'On Topic': 'The review must be about the actual product.',
        'No Repetition': 'Avoid repeating the same phrases.',
    }

    def __init__(self, trained_policy: BigramPolicy):
        self.policy = trained_policy

    def check_review(self, user_review: str) -> None:
        """Check a user's draft review and provide compliance feedback."""
        print('─' * 60)
        print(f'SUBMITTED REVIEW:\n  "{user_review}"')
        print()

        total = compute_reward(user_review)
        violations = []
        positives = []

        rule_scores = {
            'No Profanity':     reward_no_profanity(user_review),
            'No Promotional':   reward_no_promotional(user_review),
            'No URLs':          reward_no_urls(user_review),
            'No Incentivized':  reward_no_incentivized(user_review),
            'Min Length':       reward_minimum_length(user_review),
            'No ALL-CAPS':      reward_no_allcaps(user_review),
            'On Topic':         reward_on_topic(user_review),
            'No Repetition':    reward_no_repetition(user_review),
        }

        for rule, score in rule_scores.items():
            if score < -0.1:
                violations.append((rule, score))
            elif score > 0.1:
                positives.append((rule, score))

        if violations:
            print('VIOLATIONS FOUND:')
            for rule, score in violations:
                print(f'  ✗ [{score:+.2f}] {rule}: {self.RULE_EXPLANATIONS[rule]}')
        else:
            print('  No violations found.')

        if positives:
            print('COMPLIANT ASPECTS:')
            for rule, score in positives:
                print(f'  ✓ [{score:+.2f}] {rule}')

        print(f'\nOVERALL COMPLIANCE SCORE: {total:+.3f}')
        threshold = 2.0
        if total >= threshold:
            print('STATUS: COMPLIANT — Review meets Amazon guidelines.')
        else:
            print(f'STATUS: NON-COMPLIANT (score < {threshold})')
            print()
            print('SUGGESTED COMPLIANT ALTERNATIVE (generated by RL policy):')
            # Sample 5 alternatives and pick the best
            candidates = []
            for _ in range(20):
                tokens, _ = self.policy.generate(max_len=30)
                text = self.policy.tokens_to_text(tokens)
                r = compute_reward(text)
                candidates.append((r, text))
            best_r, best_text = max(candidates, key=lambda x: x[0])
            print(f'  "{best_text}"')
            print(f'  [Compliance score: {best_r:+.3f}]')
        print('─' * 60)


# Use the GRPO-trained policy (typically best)
chatbot = AmazonReviewChatbot(trained_policy=grpo.policy)

TEST_REVIEWS = [
    # Violations
    'BUY NOW!! Click www.bestdeals.com for discount! AMAZING PRODUCT!!!',
    'I got this for free in exchange for review. It\'s OK I guess.',
    'Terrible scam garbage product hate it',
    'Good.',
    # Compliant
    ('This product works great and arrived well packaged. '
     'The quality is excellent and I would highly recommend it. '
     'Very satisfied with the value and fast delivery.'),
]

for review in TEST_REVIEWS:
    chatbot.check_review(review)
    print()

## Section 10: Algorithm Comparison Summary

Visualise the probability shift: which words did the RL algorithms learn to prefer/avoid?

In [ ]:
# ─────────────────────────────────────────────
# Vocabulary Preference Analysis
# ─────────────────────────────────────────────

def marginal_token_probs(policy, n_samples=200, max_len=25):
    """Estimate marginal probability of each token across generated sequences."""
    counts = Counter()
    total = 0
    for _ in range(n_samples):
        tokens, _ = policy.generate(max_len=max_len)
        for t in tokens:
            if t not in (BOS_ID, EOS_ID):
                counts[id2token[t]] += 1
                total += 1
    return {w: c / total for w, c in counts.items()}


print('Computing marginal token probabilities...')
random_probs = marginal_token_probs(init_policy)
grpo_probs = marginal_token_probs(grpo.policy)

# Highlight good vs bad words
tracked_good = ['product', 'quality', 'excellent', 'recommend', 'great', 'satisfied', 'value', 'works']
tracked_bad = ['scam', 'fake', 'junk', 'buy', 'click', 'discount', 'free', 'garbage']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, tracked, label, color_good, color_bad in [
    (axes[0], tracked_good, 'Product-Relevant Words (should INCREASE)', '#2ecc71', '#3498db'),
    (axes[1], tracked_bad, 'Violation Words (should DECREASE)', '#e74c3c', '#e67e22'),
]:
    words = [w for w in tracked if w in random_probs or w in grpo_probs]
    x = np.arange(len(words))
    w = 0.35
    r_vals = [random_probs.get(w, 0) for w in words]
    g_vals = [grpo_probs.get(w, 0) for w in words]

    ax.bar(x - w/2, r_vals, w, label='Random Policy', color='#95a5a6', alpha=0.8)
    ax.bar(x + w/2, g_vals, w, label='GRPO Trained', color=color_good, alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(words, rotation=30, ha='right')
    ax.set_ylabel('Marginal Probability')
    ax.set_title(label)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('rl_vocab_shift.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: rl_vocab_shift.png')

## Section 11: Scaling to Real LLMs

### From Toy to Production

| Component | This Notebook | Production (e.g. GPT-4 fine-tuning) |
|-----------|--------------|--------------------------------------|
| Policy | Bigram (V×V matrix) | Transformer (billions of params) |
| Vocabulary | 60 tokens | 50,000+ tokens (BPE) |
| Reward | Rule-based functions | Learned reward model + rule functions |
| Algorithm | REINFORCE / PPO / GRPO | PPO (InstructGPT), GRPO (DeepSeek-R1) |
| Baseline | EMA of rewards | Learned value network |
| KL penalty | vs. random init | vs. supervised fine-tuned (SFT) reference |
| Training | Single machine, seconds | Multi-GPU/TPU clusters, hours-days |

### RLHF Pipeline (InstructGPT / ChatGPT)
```
1. SFT:   Supervised fine-tune on human demonstrations
2. RM:    Train reward model on human preference pairs (A better than B)
3. RLHF:  PPO with the reward model as the reward function
```

### Constitutional AI (Anthropic)
```
1. CAI critique: Model critiques its own outputs against a constitution (rules)
2. CAI revision: Model revises based on critique
3. RLAIF:  RL from AI feedback — AI model labels preference pairs (cheaper than humans)
```

### DeepSeek-R1 (GRPO for Reasoning)
```
Rules: math answer correct? format <think>...</think><answer>...</answer>?
Reward: +1 if correct, +0.1 if format compliant, 0 otherwise
Algorithm: GRPO with group size G=8, KL penalty to SFT reference
```

### Applying to Amazon Reviews at Scale
```python
# Pseudocode for real RLHF training loop
for step in range(training_steps):
    prompts = sample_product_contexts(batch_size)
    reviews = llm.generate(prompts, n_samples=group_size)  # G per prompt
    
    # Rule-based reward (deterministic)
    rule_rewards = [amazon_rule_reward(r) for r in reviews]
    
    # Optionally: learned reward model for nuanced quality
    quality_rewards = reward_model(prompts, reviews)
    
    total_rewards = rule_rewards + 0.5 * quality_rewards
    
    # GRPO update
    advantages = group_normalise(total_rewards, group_size)
    loss = grpo_clipped_loss(llm, ref_llm, reviews, advantages, kl_coef=0.01)
    loss.backward()
    optimizer.step()
```

In [ ]:
# ─────────────────────────────────────────────
# Final Summary
# ─────────────────────────────────────────────

print('=' * 65)
print('  RL FOR RULE-FOLLOWING: AMAZON REVIEW COMPLIANCE CHATBOT')
print('=' * 65)

print('''
ALGORITHMS IMPLEMENTED:
  1. REINFORCE — Policy gradient, 1 episode/step, EMA baseline
  2. PPO       — Batch episodes, clipped ratio, K epochs/batch
  3. GRPO      — Group normalised advantage, KL-to-reference penalty
                 (the algorithm behind DeepSeek-R1)

AMAZON RULES ENCODED AS REWARD COMPONENTS:
  ✗ No profanity / offensive language
  ✗ No promotional / advertising content
  ✗ No external URLs
  ✗ No incentivized review disclosure
  ✓ Minimum substantive length (15 words)
  ✗ No ALL-CAPS usage
  ✓ On-topic product content
  ✗ No repetitive phrases

KEY INSIGHT:
  Rules → Reward functions → RL training → Policy that follows rules
  No labelled data required. Add a new rule by writing a new reward fn.

PRODUCTION PATHWAY:
  This notebook (bigram, 60 tokens) → Real LLM (transformer, 50K tokens)
  Replace BigramPolicy with a pre-trained language model
  Replace toy vocabulary with full BPE tokenization
  Add learned reward model for nuanced quality signals
  Deploy as review validation + generation API
''')

final_scores = {
    'Random (untrained)': baseline_eval['total'],
    'REINFORCE':          reinforce_eval['total'],
    'PPO':                ppo_eval['total'],
    'GRPO':               grpo_eval['total'],
}
print('FINAL COMPLIANCE SCORES (avg over 100 samples):')
for name, score in final_scores.items():
    improvement = score - final_scores['Random (untrained)']
    bar = '█' * max(0, int(score * 3))
    print(f'  {name:<22} {score:+.3f}  Δ{improvement:+.3f}  {bar}')